# Walkthrough del sistema RAG — paso a paso

Este notebook recorre, **pieza a pieza**, el sistema RAG del TFM construido desde
cero. Cada paso tiene tres partes: **qué hace y por qué**, el **código real** (que
se importa de `src/`, no se duplica aquí) y su **resultado**.

## ¿Qué es un RAG?

Un LLM solo sabe lo que aprendió durante su entrenamiento. Un sistema **RAG**
(*Retrieval-Augmented Generation*, generación aumentada por recuperación) le
da, en cada pregunta, los fragmentos de un documento que hacen falta para
responderla:

```
                 PREPARACIÓN (una vez)                          CADA PREGUNTA
 ┌─────┐   ┌─────────┐   ┌────────┐   ┌──────────┐   ┌───────┐
 │ PDF │ → │Markdown │ → │ chunks │ → │embeddings│ → │ índice│ ← pregunta → embedding
 └─────┘   └─────────┘   └────────┘   └──────────┘   └───────┘
  Docling   jerarquía     trocear      Gemini         FAISS      los k chunks más parecidos
            de títulos                                            + pregunta → LLM → respuesta con cita
```

| Sección | Pieza | Día |
|---|---|---|
| 1 | Ingesta: PDF → Markdown con páginas y jerarquía de títulos | 1 |
| 2 | Loader: páginas limpias y corpus | 1 |
| 3 | Chunking: trocear en fragmentos con metadatos | 1 |
| 4 | Embeddings: convertir texto en vectores | 2 |
| 5 | Similitud coseno a mano y comprobación con FAISS | 2 |
| 6 | Búsqueda: los fragmentos más parecidos a una pregunta | 2 |

**Documento**: *Vampiro: La Mascarada*, 5ª edición. Elegido por ser un reglamento
complejo, con tablas, dos columnas y reglas que se cruzan entre capítulos.

> **Antes de ejecutar.** Usa el kernel del entorno virtual del proyecto
> (`.venv`, Python 3.11). El notebook lee lo ya calculado en `data/processed/`
> y `storage/`, así que **no repite la conversión ni gasta cuota de la API**
> (solo la sección 6 hace una llamada si cambias la pregunta).
>
> **Copyright.** El notebook se guarda en el repositorio **sin salidas**,
> porque estas contienen texto del libro. Ejecútalo en local para verlas.

### Preparación

El notebook vive en `notebooks/`, pero el código está en `src/`. La primera
celda le dice a Python dónde está la raíz del proyecto para poder importarlo.

In [ ]:
import sys
from pathlib import Path

# Raíz del proyecto: la carpeta que contiene notebooks/ y src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import re
import statistics

import matplotlib.pyplot as plt

from src import config

print("Raíz del proyecto:", config.RAIZ)

# Estilo de los gráficos: rejilla discreta, sin bordes superfluos.
AZUL, NARANJA, TINTA_2, REJILLA = "#2a78d6", "#eb6834", "#52514e", "#e6e5e0"

def estilo_ejes(ax):
    """Aspecto común de los gráficos: solo los ejes necesarios, rejilla suave."""
    for borde in ("top", "right"):
        ax.spines[borde].set_visible(False)
    for borde in ("left", "bottom"):
        ax.spines[borde].set_color(TINTA_2)
    ax.tick_params(colors=TINTA_2)
    ax.grid(axis="y", color=REJILLA, linewidth=0.8)
    ax.set_axisbelow(True)

---
## 1. Ingesta: del PDF a un Markdown con páginas y títulos

### 1.1 Conversión con Docling (`src/ingest/convert_docling.py`)

Un PDF no guarda "texto": guarda letras colocadas en posiciones de la página.
Extraerlo bien es difícil cuando hay **tablas** y **dos columnas**. En el punto
de control del día 1 (experimento E1) se compararon dos extractores:

- **pypdf**: rápido, pero rompe las tablas (cada fila acaba partida en 3-4 líneas).
- **Docling**: analiza la maquetación con modelos de visión y reconstruye tablas,
  columnas y títulos como Markdown.

Se eligió **Docling**, ejecutado **una sola vez** para todo el libro (404
páginas, 6,5 minutos). Dos decisiones importantes:

1. **Marcador de página.** Antes de cada página se escribe `<!-- pagina: N -->`
   con la **página impresa** (la que vería un lector en el libro; en este PDF
   es la página del PDF menos 2). Así cada fragmento sabrá de qué página viene
   y la respuesta podrá **citarla**.
2. Se guarda también el resultado completo en `.json`, para poder regenerar el
   Markdown sin repetir la conversión.

Veamos una página tal como sale de Docling:

In [ ]:
def pagina_de(markdown, numero):
    """Texto de una página impresa concreta de un Markdown con marcadores."""
    trozos = re.split(r"<!-- pagina: (\d+) -->", markdown)
    for n, texto in zip(trozos[1::2], trozos[2::2]):
        if int(n) == numero:
            return texto.strip()

md_docling = next(config.CARPETA_PROCESADOS.glob("*.md")).read_text(encoding="utf-8")
print(f"Markdown de Docling: {len(md_docling):,} caracteres, "
      f"{md_docling.count('<!-- pagina:')} marcadores de página\n")
print(pagina_de(md_docling, 254)[:700])

### 1.2 El problema: todos los títulos al mismo nivel

Docling marca **todos** los títulos con `##`. Una disciplina entera
("Celeridad"), una parte de ella ("Nivel 4") y un recuadro ("EJEMPLO:") quedan
al mismo nivel, y los nombres de los poderes (en versalitas, pegados al párrafo)
ni siquiera se detectan como títulos. Así queda el "árbol" de las páginas de
Celeridad según Docling:

In [ ]:
def arbol(markdown, desde, hasta):
    """Imprime los títulos de un rango de páginas, sangrados según su nivel."""
    trozos = re.split(r"<!-- pagina: (\d+) -->", markdown)
    for n, texto in zip(trozos[1::2], trozos[2::2]):
        if desde <= int(n) <= hasta:
            for linea in texto.splitlines():
                m = re.match(r"(#+) (.*)", linea)
                if m:
                    nivel = len(m.group(1))
                    print(f"pág. {n:>3}   " + "    " * (nivel - 1) + m.group(2))

arbol(md_docling, 252, 254)

### 1.3 La solución: la jerarquía está en la tipografía (`src/ingest/estructura.py`)

El libro distingue cada nivel de título con una **fuente y un tamaño** concretos.
Un análisis de todas las fuentes del corpus (experimento E3) encontró 5 estilos
de título consistentes en todos los capítulos. Ese **perfil tipográfico** se
declara en `config.py` (el código no sabe nada de Vampiro: para otro PDF solo
cambiaría el perfil):

In [ ]:
info_documento = next(iter(config.DOCUMENTOS.values()))
for (fuente, tamano), nivel in info_documento["titulos"].items():
    print(f"nivel {nivel}: {fuente:<32} {tamano:>3} pt")

`estructura.py` lee con pypdf la fuente y el tamaño de cada fragmento de texto,
localiza los títulos de cada página y los coloca en el Markdown de Docling con
su nivel real (`#`, `##`, `###`...). Para emparejar el texto de pypdf con el de
Docling usa una **comparación normalizada** (minúsculas, sin tildes, solo letras
y números), que no se rompe por espacios o guiones distintos. Tras varias
iteraciones (experimentos E5 y E8) coloca 875 de 895 títulos, y **ninguno**
queda sin colocar dentro del corpus.

El mismo rango de páginas, ahora con la jerarquía reconstruida:

In [ ]:
md_estructurado = next(config.CARPETA_ESTRUCTURADOS.glob("*.md")).read_text(encoding="utf-8")
arbol(md_estructurado, 252, 254)

Ahora se lee la estructura lógica del libro: **Celeridad > Nivel 4 > Puntería
certera**. (La primera línea, «Telepatía», es el último poder de Auspex, la
disciplina anterior, que termina en la pág. 252.) Esto será clave para trocear por secciones con sentido y para que
cada fragmento "sepa" dónde está.

---
## 2. Loader: páginas limpias y corpus (`src/ingest/loader.py`)

El loader lee los Markdown, los parte por los marcadores de página y devuelve
una lista de **páginas**, cada una con su documento, su número, su texto limpio
y si forma parte del **corpus**.

**¿Por qué un corpus?** El libro tiene 404 páginas, pero el TFM se centra en
tres temas (creación de personaje, combate y disciplinas, la "magia" del juego).
Los rangos de páginas están en `config.py`. Un detalle: las páginas de fuera del
corpus **no se descartan**, solo se marcan, porque sus títulos hacen falta para
la jerarquía (el título del capítulo "Personajes" está en la pág. 133, fuera del
corpus, pero sus secciones empiezan en la 135).

In [ ]:
from src.ingest.loader import cargar_documentos

paginas = cargar_documentos()
corpus = [p for p in paginas if p.en_corpus]
print(f"Páginas cargadas: {len(paginas)}  |  en el corpus: {len(corpus)}")
print("Rangos del corpus (páginas impresas):", info_documento["rangos_paginas"])
print(f"Caracteres en el corpus: {sum(len(p.texto) for p in corpus):,}")

### 2.1 Limpieza

Docling deja algunos restos de maquetación. La revisión manual de chunks
(experimento E8) llevó a estas reglas, cada una con su motivo:

| Regla | Ejemplo | Motivo |
|---|---|---|
| Quitar marcadores de imagen y viñetas decorativas | `<!-- image -->`, `- ■ ■ Coste:` | No aportan contenido |
| Unir palabras partidas por un salto de columna | `se resuel-` / `ve` | El texto del libro está maquetado en columnas |
| Recuperar el punto del nivel 1 en las listas de niveles | `- Conoces...` → `- · Conoces...` | Docling convierte el `•` en una viñeta |
| Quitar texto decorativo | `2 4 8`, `ejecutivo Ejecutivo` | Números de página y rótulos de ilustraciones |

Un ejemplo construido a mano, con un caso de cada regla:

In [ ]:
from src.ingest.loader import limpiar

sucio = """<!-- image -->
- ■ ■ Coste: Un control de Enardecimiento.

El combate a distancia se resuel-

ve como una contienda.

- Conoces los senderos.
- ·· Pasas más tiempo fuera que dentro.

2 4 8
ejecutivo Ejecutivo"""

print("ANTES:\n" + sucio)
print("\nDESPUÉS:\n" + limpiar(sucio))

Otra regla mira **varias páginas seguidas**: quita las etiquetas en negrita que
interrumpen una frase a medias (rótulos de ilustraciones que Docling coloca en
mitad del texto), aunque la frase empiece en la página anterior:

In [ ]:
from src.ingest.loader import Pagina, quitar_etiquetas_flotantes

ejemplo = [
    Pagina("doc", "Documento", 140, "...¿Intentarás volver a ser humano o secarás a un", True),
    Pagina("doc", "Documento", 141, "**Investigador investigador**\n\nCainita para abrazar la noche?", True),
]
print("Etiquetas quitadas:", quitar_etiquetas_flotantes(ejemplo))
print("Página 141 tras la limpieza:", repr(ejemplo[1].texto.strip()))

---
## 3. Chunking: trocear en fragmentos (`src/chunking/chunker.py`)

No se puede buscar en el libro entero de una vez: se trocea en **chunks**
(fragmentos), y cada uno se convertirá en un vector. El tamaño importa: si es
pequeño, el fragmento pierde contexto; si es grande, mezcla temas y su vector
se "diluye". Hay dos estrategias implementadas:

1. **Tamaño fijo con overlap (baseline).** Ventanas de 1000 caracteres que se
   solapan 200. Es la técnica más simple; no mira el contenido.
2. **Estructural híbrida (la principal).** Un chunk por **sección lógica** (según
   los títulos del punto 1). Si una sección pasa de 3000 caracteres, se
   subdivide con el troceado de tamaño fijo (overlap de 400); si tiene menos de
   300, se une a la siguiente (si es su hija o su hermana).

### 3.1 El troceado de tamaño fijo, en pequeño

Con un texto inventado y ventanas diminutas (60 caracteres, solape de 20) se ve
cómo funciona:

- Cada trozo **termina en el mejor punto de corte** del último 25 % de la
  ventana: fin de párrafo, si no fin de línea, si no fin de frase y, en último
  caso, un espacio (nunca a mitad de palabra).
- El siguiente trozo **retrocede hasta 20 caracteres** (el solape) y avanza hasta
  el primer principio de frase o de línea de esa zona (si no hay, de palabra).

En la salida se ven los dos casos. Entre el 1.º y el 2.º no hay texto repetido:
el corte cayó justo en un fin de frase y no había otro principio de frase en la
zona de solape. Entre el 2.º y el 3.º sí se repite «la Bestia siempre»: el corte
cayó a mitad de frase y el solape conserva el contexto. Con el solape real (200
o 400 caracteres) casi siempre hay algún principio de frase en la zona.

In [ ]:
from src.chunking.chunker import trocear_fijo

texto = ("La Bestia ansía sangre. El vampiro resiste con su Voluntad. "
         "Pero cada noche cuesta más. Al final, la Bestia siempre encuentra un hueco.")
for inicio, fin in trocear_fijo(texto, tamano=60, overlap=20):
    print(f"[{inicio:>3}-{fin:>3}]  {texto[inicio:fin]!r}")

### 3.2 Las dos estrategias sobre el corpus real

In [ ]:
from src.chunking.chunker import chunks_estructurales, chunks_fijos

fijos = chunks_fijos(paginas)
estructurales, almacen = chunks_estructurales(paginas)

for nombre, chunks in (("Tamaño fijo 1000/200", fijos), ("Estructural híbrido", estructurales)):
    t = sorted(len(c.texto) for c in chunks)
    print(f"{nombre:<22} {len(chunks):>4} chunks | mediana {statistics.median(t):>5.0f} car. "
          f"| mín {t[0]:>4} | máx {t[-1]:>4}")

La diferencia se ve mejor en la distribución de tamaños. El fijo concentra casi
todos los chunks cerca de 1000 caracteres (lo marca la ventana, no el
contenido); el estructural varía con el tamaño real de cada sección, con un tope
de 3000:

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
intervalos = range(0, 3100, 100)
for ax, chunks, color, titulo in (
        (ejes[0], fijos, AZUL, f"Tamaño fijo 1000/200 ({len(fijos)} chunks)"),
        (ejes[1], estructurales, NARANJA, f"Estructural híbrido, máx. 3000 ({len(estructurales)} chunks)")):
    ax.hist([len(c.texto) for c in chunks], bins=intervalos, color=color,
            edgecolor="white", linewidth=1.5)
    ax.set_title(titulo, loc="left", fontsize=11)
    ax.set_ylabel("Nº de chunks", color=TINTA_2)
    estilo_ejes(ax)
ejes[1].set_xlabel("Tamaño del chunk (caracteres)", color=TINTA_2)
fig.suptitle("Distribución de tamaños de chunk según la estrategia", x=0.01, ha="left", fontsize=13)
fig.tight_layout()
plt.show()

### 3.3 Un chunk con todos sus metadatos

Cada chunk estructural lleva, además del texto:

- **`documento`, `pagina_inicio`, `pagina_fin`**: para citar la fuente.
- **`ruta`**: los títulos de todos los niveles, del capítulo hacia abajo. Así el
  fragmento "sabe" dónde está aunque su texto no lo diga.
- **`seccion_id` y `ancestros_ids`**: apuntan al **almacén de secciones**. El día 3
  servirán para pasar al LLM la sección padre completa (expansión al padre).

In [ ]:
chunk = next(c for c in estructurales if "Puntería certera" in c.ruta)

print("id:           ", chunk.id)
print("documento:    ", chunk.documento)
print("páginas:      ", chunk.pagina_inicio, "-", chunk.pagina_fin)
print("ruta:         ", " > ".join(chunk.ruta))
print("seccion_id:   ", chunk.seccion_id)
print("ancestros_ids:", chunk.ancestros_ids)
print("\n--- texto ---")
print(chunk.texto)

### 3.4 Los ancestros: el "contexto completo de cada nivel"

Con el almacén de secciones se puede subir por el árbol desde el chunk y ver
cuánto ocupa cada nivel completo (con todas sus subsecciones). El día 3, si se
recupera este chunk, al LLM se le podrá entregar **el ancestro más grande que
quepa en 6000 caracteres** (experimento E6 y P4):

In [ ]:
def tamano_total(seccion_id):
    """Caracteres de una sección contando todas sus subsecciones."""
    s = almacen[seccion_id]
    propio = sum(len(linea) + 1 for _, linea in s.lineas)
    return propio + sum(tamano_total(h) for h in s.hijos_ids)

for sid in chunk.ancestros_ids + [chunk.seccion_id]:
    s = almacen[sid]
    total = tamano_total(sid)
    print(f"nivel {s.nivel}  {s.titulo:<20} {total:>8,} car.   "
          f"{'cabe' if total <= 6000 else 'NO cabe'} en el tope de 6000")

---
## 4. Embeddings: de texto a vectores (`src/embeddings/embedder.py`)

### 4.1 ¿Qué es un embedding?

Un **embedding** es una lista de números que representa el **significado** de un
texto. El modelo (`gemini-embedding-2`) está entrenado para que textos con
significado parecido produzcan listas parecidas. Buscar "fragmentos que hablen
de lo mismo que la pregunta" se convierte en buscar "vectores parecidos al de
la pregunta".

- **Dimensión: 768.** El modelo produce por defecto 3072 números, pero está
  entrenado con la técnica *Matryoshka* (como las muñecas rusas): los primeros
  números concentran lo más importante, así que se pueden pedir solo 768 con una
  pérdida mínima de calidad. Es una de las tallas recomendadas por Google.
- **Prefijos de tarea.** El modelo necesita saber si un texto es una pregunta o
  un documento. Se le indica con un prefijo, en el formato de la documentación
  oficial. Para los chunks, el campo `title` es **su ruta de títulos**: el
  contexto jerárquico entra en el vector.

Esto es exactamente lo que se envía al modelo:

In [ ]:
from src.embeddings.embedder import embeber, texto_consulta, texto_documento

print("CHUNK:\n" + texto_documento(chunk)[:350] + " [...]\n")
print("PREGUNTA:\n" + texto_consulta("¿Qué hace el poder Puntería certera de Celeridad?"))

### 4.2 Un vector real

El vector del chunk anterior (sale de la caché, no llama a la API). Sus números,
por sí solos, no significan nada para una persona: lo que importa es cómo se
comparan con otros vectores. Su **norma** (longitud) es 1: el modelo ya lo
devuelve normalizado, algo que en la sección 5 veremos por qué importa.

In [ ]:
from src.retrieval.cosine_manual import norma

vector = embeber([texto_documento(chunk)])[0]
print("dimensión:        ", len(vector))
print("primeros 8 números:", [round(x, 4) for x in vector[:8]])
print("norma:            ", round(norma(vector), 6))

### 4.3 La caché en disco

Cada llamada a la API cuesta tiempo y cuota (en el tier gratuito, unos 100
textos por minuto: experimento E11). Por eso cada vector se guarda en
`storage/cache_embeddings.json`, con una **clave** que es el hash SHA-256 de
(modelo, dimensión, texto). Si cambia cualquiera de los tres, la clave cambia y
se recalcula; si no, nunca se repite una llamada. Reconstruir el índice entero
ya no gasta cuota.

In [ ]:
from src.embeddings.embedder import clave_cache

cache = json.loads(config.CACHE_EMBEDDINGS.read_text(encoding="utf-8"))
print("clave del chunk anterior:", clave_cache(texto_documento(chunk)))
print(f"vectores en caché: {len(cache)}  |  archivo: "
      f"{config.CACHE_EMBEDDINGS.stat().st_size / 1e6:.1f} MB")

---
## 5. Similitud coseno, a mano (`src/retrieval/cosine_manual.py`)

### 5.1 La fórmula

Para comparar dos vectores se usa la **similitud coseno**, que mide el **ángulo**
entre ellos, no su longitud:

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}
\qquad a \cdot b = \sum_{i} a_i \, b_i \qquad \lVert a \rVert = \sqrt{\sum_i a_i^2}$$

- **1**: misma dirección (mismo significado).
- **0**: perpendiculares (nada que ver).
- **−1**: direcciones opuestas.

`cosine_manual.py` implementa las tres operaciones con bucles de Python puro,
**sin numpy**, para que se vea que no hay magia. Primero, un ejemplo con
vectores de solo 3 números, calculado paso a paso:

In [ ]:
from src.retrieval.cosine_manual import ejemplo_juguete

ejemplo_juguete()

### 5.2 Por qué FAISS con `IndexFlatIP` y vectores normalizados

FAISS es una librería de búsqueda de vectores. Su índice `IndexFlatIP` calcula
el **producto escalar** ("inner product", IP) de la pregunta con **todos** los
vectores (búsqueda exacta, sin aproximaciones). El producto escalar *no* es el
coseno... salvo cuando los vectores tienen norma 1, porque entonces el
denominador de la fórmula vale 1:

$$\lVert a \rVert = \lVert b \rVert = 1 \;\Rightarrow\; \cos(a, b) = a \cdot b$$

Por eso **todos los vectores se normalizan** antes de añadirlos al índice y
antes de buscar. Para comprobarlo, el coseno a mano y FAISS sobre los 324
chunks reales, con 3 preguntas (una por capítulo):

In [ ]:
from src.retrieval.cosine_manual import comparar_con_faiss

comparar_con_faiss()

Mismos chunks, mismo orden y las mismas puntuaciones hasta la séptima cifra
decimal (FAISS calcula con números de 32 bits y Python con 64). **FAISS no hace
nada distinto de lo que hemos programado a mano: solo lo hace mucho más rápido.**

### 5.3 Una observación importante: los cosenos son todos "altos"

Si se calcula el coseno de una pregunta con **todos** los chunks, ni los
fragmentos sin ninguna relación bajan mucho de 0,5. Es habitual en los modelos
de embeddings (los vectores no se reparten por todas las direcciones posibles,
lo que se llama *anisotropía*). Consecuencia práctica: lo que importa es el
**orden** de las puntuaciones, no su valor absoluto. Lo tendremos en cuenta en
el router del día 4, donde un umbral fijo habrá que calibrarlo con datos.

In [ ]:
from src.embeddings.embedder import embeber_consulta
from src.retrieval.cosine_manual import similitud_coseno
from src.vectorstore.faiss_store import IndiceVectorial

indice = IndiceVectorial.cargar("estructural")
pregunta = "¿Qué hace el poder Puntería certera de Celeridad?"
v_pregunta = embeber_consulta(pregunta)   # en caché: no llama a la API
similitudes = [similitud_coseno(v_pregunta, list(map(float, indice.indice.reconstruct(i))))
               for i in range(indice.indice.ntotal)]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(similitudes, bins=40, color=AZUL, edgecolor="white", linewidth=1.5)
mejor = max(similitudes)
ax.axvline(mejor, color=TINTA_2, linestyle="--", linewidth=1.2)
ax.annotate(f"chunk correcto\n(Puntería certera): {mejor:.2f}", xy=(mejor, ax.get_ylim()[1] * 0.75),
            xytext=(-10, 0), textcoords="offset points", ha="right", color=TINTA_2, fontsize=9)
ax.set_xlabel("Similitud coseno con la pregunta", color=TINTA_2)
ax.set_ylabel("Nº de chunks", color=TINTA_2)
ax.set_title(f"«{pregunta}» frente a los {len(similitudes)} chunks", loc="left", fontsize=11)
estilo_ejes(ax)
fig.tight_layout()
plt.show()
print(f"mínimo {min(similitudes):.3f} | mediana {statistics.median(similitudes):.3f} | máximo {mejor:.3f}")

El gráfico tiene **tres escalones**, y no es casualidad:

1. **0,81**: el chunk correcto, «Celeridad > Nivel 4 > Puntería certera».
2. **0,67-0,72**: un pequeño grupo que son **todos los demás poderes de
   Celeridad** (Reflejos rápidos, Gracia felina, Golpe relámpago...).
3. **0,49-0,65**: el resto del libro.

El vector captura la **jerarquía**: los fragmentos de la misma disciplina quedan
más cerca de la pregunta que el resto, en buena parte gracias a la ruta de
títulos que se pasa como `title` al calcular el embedding.

---
## 6. Búsqueda: la "R" de RAG (`src/vectorstore/faiss_store.py`)

Con todas las piezas, buscar es: **embedding de la pregunta → FAISS → los k
chunks más parecidos**, cada uno con su ruta y sus páginas para poder citarlo.
Cambia la pregunta y vuelve a ejecutar la celda (la primera vez que se usa una
pregunta nueva se hace una llamada a la API; después queda en caché).

El día 3 se añadirá la "G" de RAG: estos fragmentos y la pregunta irán al LLM,
que redactará la respuesta citando documento y página.

In [ ]:
PREGUNTA = "¿Puedo gastar Fuerza de Voluntad para volver a tirar dados?"

resultados = indice.buscar(embeber_consulta(PREGUNTA), k=5)
print(f"Pregunta: {PREGUNTA}\n")
for puesto, (c, similitud) in enumerate(resultados, 1):
    paginas_txt = (f"pág. {c.pagina_inicio}" if c.pagina_inicio == c.pagina_fin
                   else f"págs. {c.pagina_inicio}-{c.pagina_fin}")
    print(f"{puesto}. [{similitud:.3f}] {' > '.join(c.ruta)}  ({paginas_txt})")
    print("   " + c.texto[:220].replace("\n", " ") + " ...\n")

---
## Resumen de los días 1 y 2

| Pieza | Resultado |
|---|---|
| Conversión (Docling) | 404 páginas, tablas reconstruidas, marcador de página impresa |
| Jerarquía de títulos | 5 niveles a partir de la tipografía; 875/895 títulos colocados |
| Loader | 112 páginas de corpus, limpieza de restos de maquetación |
| Chunking estructural | 324 chunks con ruta de títulos, páginas y ancestros |
| Embeddings | `gemini-embedding-2`, 768 dimensiones, prefijos de tarea, caché en disco |
| Coseno a mano vs. FAISS | Mismo resultado (diferencia máxima ~1e-7) |

Todos los análisis, con sus datos, están en el **Registro de experimentos** de
`docs/memoria/04_implementacion.md` (E1-E11).

**Siguiente (día 3):** retriever top-k, generación con citas y pipeline completo.